In [ ]:
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from omegaconf import DictConfig
import lightning as L
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor

# Ensure project root is in path for imports
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.timeseries.data.pems import PeMS08DataModule,PeMS08,PeMS08MultiScaleDataModule
from src.timeseries.lejepa import LeJEPA_Forecaster
from src.timeseries.models.cnn import TimeSeriesEncoder
from src.timeseries.visualizations.datamodule import visualize_pems_tuple
from src.timeseries.visualizations.callbacks import VisualizationCallback
import matplotlib.pyplot as plt

In [ ]:
cfg = DictConfig({
    'window_size': 96,
    'target_window_size': 12,
    'temporal_shift': 12,
    'stride': 1,
    'batch_size': 128,
    'num_workers': 16,
    'lr': 3e-4,
    'lamb': .,
    'epochs': 800,
    'accelerator': 'auto',
    
    'repeat_factor': 15,
    # Time-series CV
    # "split_mode": "ts_cv",
    # "cv_folds": 5,        # number of folds
    # "cv_fold": 0,         # which fold to run (0..cv_folds-1)
    # "cv_gap": 0,          # optional gap between train and val (reduces leakage)
    # # "cv_test_size": 20000,  # optional; otherwise auto = T//(cv_folds+1)
})

In [ ]:
pems=PeMS08()
pems.target

In [ ]:
import plotly.express as px
n=100
px.line(pems.target[pems.target.columns[n:n+10]])

In [ ]:
datamodule = PeMS08DataModule(cfg)
datamodule.prepare_data()
datamodule.setup()

In [ ]:
train_datalaoder=datamodule.train_dataloader()
for batch in train_datalaoder:
    #print(batch)
    print(batch[0].shape)
    break    

In [ ]:
# batch,view,nodes,time

In [ ]:
import plotly.express as px
px.line(batch[0][0,0,n:n+10,:].reshape(96,10).to("cpu").numpy())

In [ ]:
visualize_pems_tuple(datamodule, stage='train', batch_index=0, sensor_index=0)

In [ ]:
visualize_pems_tuple(datamodule, stage='train', batch_index=0, sensor_index=0)

In [ ]:
visualize_pems_tuple(datamodule, stage='train', batch_index=0, sensor_index=0)

In [ ]:
visualize_pems_tuple(datamodule, stage='train', batch_index=0, sensor_index=170)

In [ ]:
from src.timeseries.model.rnn import TimeSeriesLSTMEncoder
encoder = TimeSeriesLSTMEncoder(input_channels=170, output_dim=512)
model = LeJEPA_Forecaster(
    encoder_backbone=encoder,
    input_dim=170,
    horizon=12,
    proj_dim=12,
    lamb=cfg.lamb,
    lr=cfg.lr,
    scaler_mean=datamodule.scaler.mean,
    scaler_std=datamodule.scaler.std,
    reg_type="sigreg", 
    
)

print("Model initialized correctly with scaler info for real-scale MAE.")

In [ ]:
logger = TensorBoardLogger("tb_logs", name="lejepa_pems08")

callbacks = [
    ModelCheckpoint(
        dirpath="checkpoints",
        filename="lejepa-{epoch:02d}-{val/mae_unscaled:.4f}",
        monitor="val/mae_unscaled",
        mode="min",
        save_top_k=3
    ),
    LearningRateMonitor(logging_interval="step"),
    VisualizationCallback(num_samples_plot=3, umap_every_n_epochs=1, num_umap_batches=10)
]

trainer = L.Trainer(
    max_epochs=cfg.epochs,
    accelerator=cfg.accelerator,
    devices=1,
    logger=logger,
    callbacks=callbacks,
    log_every_n_steps=10,
    reload_dataloaders_every_n_epochs=10,
    gradient_clip_val=.30,
    gradient_clip_algorithm="norm",
    
)

print("Starting training...")


In [ ]:
# # cfg = DictConfig({
#     'window_size': 96,
#     'target_window_size': 12,
#     #'temporal_shift': 24,
#     'stride': 1,
#     'batch_size': 128,
#     'num_workers': 16,
#     'lr': 5e-3,
#     'lamb': 0.7,
#     'epochs': 800,
#     'accelerator': 'auto',
#     # Time-series CV
#     # "split_mode": "ts_cv",
#     # "cv_folds": 5,        # number of folds
#     # "cv_fold": 0,         # which fold to run (0..cv_folds-1)
#     # "cv_gap": 0,          # optional gap between train and val (reduces leakage)
#     # # "cv_test_size": 20000,  # optional; otherwise auto = T//(cv_folds+1)
# Cyclical LR
# }) --> 24
# # cfg = DictConfig({
#     'window_size': 96,What wo
#     'target_window_size': 12,
#     #'temporal_shift': 24,
#     'stride': 1,
#     'batch_size': 128,
#     'num_workers': 16,
#     'lr': 3e-5,
#     'lamb': 0.7,
#     'epochs': 800,
#     'accelerator': 'auto',
#     # Time-series CV
#     # "split_mode": "ts_cv",
#     # "cv_folds": 5,        # number of folds
#     # "cv_fold": 0,         # which fold to run (0..cv_folds-1)
#     # "cv_gap": 0,          # optional gap between train and val (reduces leakage)
#     # # "cv_test_size": 20000,  # optional; otherwise auto = T//(cv_folds+1)
# }) -->

In [ ]:
trainer.fit(model, datamodule=datamodule)